# 03 — Comprehensions and Iteration Patterns

Comprehensions are Python's signature one-liner for building lists, dicts, and sets from an iterable. Used well, they're *clearer* than a loop. Used poorly, they're a wall of text.

Rule of thumb: if your comprehension has **more than one `for`** *and* an `if`, break it into a loop.

## List comprehensions

Shape: `[expr for item in iterable if condition]`

Read left-to-right: "give me `expr` for each `item` in `iterable`, where `condition`."

In [1]:
# loop version
squares = []
for n in range(10):
    squares.append(n * n)
print(squares)

# comprehension — same result
squares = [n * n for n in range(10)]
print(squares)

# with a filter
even_squares = [n * n for n in range(10) if n % 2 == 0]
print(even_squares)

# transform + filter
words = ["hello", "hi", "hey", "greetings"]
short_upper = [w.upper() for w in words if len(w) <= 3]
print(short_upper)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[0, 4, 16, 36, 64]
['HI', 'HEY']


### Conditional *expression* inside a comprehension

Don't confuse the **filter** (`if` at the end) with a **conditional expression** (`a if cond else b` in the value position).

In [2]:
# Replace negatives with 0, keep others as-is
xs = [3, -1, 4, -2, 5]
clipped = [x if x >= 0 else 0 for x in xs]
print(clipped)

# vs. dropping negatives entirely
kept = [x for x in xs if x >= 0]
print(kept)

[3, 0, 4, 0, 5]
[3, 4, 5]


### Nested comprehensions — the flatten pattern

In [3]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# Flatten:  read the `for`s LEFT to RIGHT, same order as nested loops
flat = [x for row in matrix for x in row]
print(flat)

# Equivalent loop:
flat2 = []
for row in matrix:
    for x in row:
        flat2.append(x)
print(flat2)

[1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2, 3, 4, 5, 6, 7, 8, 9]


## Dict comprehensions

Shape: `{key_expr: value_expr for item in iterable if condition}`

In [4]:
# Build a lookup from a list
words = ["apple", "banana", "cherry"]
lengths = {w: len(w) for w in words}
print(lengths)

# Invert a dict (assumes values are unique)
inv = {v: k for k, v in lengths.items()}
print(inv)

# Filter a dict
scores = {"alice": 90, "bob": 55, "cleo": 78, "dan": 40}
passing = {name: s for name, s in scores.items() if s >= 60}
print(passing)

{'apple': 5, 'banana': 6, 'cherry': 6}
{5: 'apple', 6: 'cherry'}
{'alice': 90, 'cleo': 78}


## Set comprehensions

In [5]:
text = "the quick brown fox jumps over the lazy dog"
first_letters = {w[0] for w in text.split()}
print(first_letters)

{'t', 'b', 'l', 'j', 'q', 'o', 'f', 'd'}


## Generator expressions — the lazy cousin

Same syntax with `()` instead of `[]`. Doesn't build the full result in memory — yields items one at a time. Perfect feed for `sum`, `any`, `all`, `max`, `min`.

In [6]:
# sum of squares up to 1 million — no million-element list created
print(sum(n * n for n in range(1_000_000)))

# any() / all() short-circuit
nums = [2, 4, 6, 7, 8]
print(any(n % 2 == 1 for n in nums))   # True (7)
print(all(n % 2 == 0 for n in nums))   # False

# a gen-exp passed as the SOLE argument to a function can skip the outer parens:
print(sum(n for n in nums if n > 3))

333332833333500000
True
False
25


We'll go deeper on generators in [07_iterators_generators](../07_iterators_generators/) — for now, just know that *gen-exp = comprehension that doesn't build the list*.

## Iteration patterns you'll use forever

### `enumerate` — index + value

In [7]:
for i, name in enumerate(["alice", "bob", "cleo"], start=1):
    print(f"{i}. {name}")

1. alice
2. bob
3. cleo


### `zip` — walk multiple sequences in parallel

Use `strict=True` (3.10+) to assert the sequences are the same length — silent truncation is a common bug.

In [8]:
names  = ["alice", "bob", "cleo"]
scores = [90, 75, 60]
weights = [1.0, 0.5, 1.0]

for name, score, weight in zip(names, scores, weights, strict=True):
    print(f"{name}: weighted = {score * weight}")

# zip(*matrix) transposes — surprising and useful
matrix = [[1, 2, 3], [4, 5, 6]]
for row in zip(*matrix):
    print(row)

alice: weighted = 90.0
bob: weighted = 37.5
cleo: weighted = 60.0
(1, 4)
(2, 5)
(3, 6)


### Unpacking — `*` and `**` in calls and assignments

Quick preview — we cover this fully in `02_functions_deep`.

In [9]:
first, *rest = [1, 2, 3, 4, 5]
print(first, rest)

*init, last = [1, 2, 3, 4, 5]
print(init, last)

a, *mid, b = [1, 2, 3, 4, 5]
print(a, mid, b)

1 [2, 3, 4, 5]
[1, 2, 3, 4] 5
1 [2, 3, 4] 5


## Mini-exercise

1. One-liner: from `range(1, 31)`, produce `["Fizz", 2, "Buzz", 4, "Fizz", ...]` (FizzBuzz as a list comprehension using a conditional expression).
2. Given `words = ["apple", "banana", "avocado", "cherry", "blueberry"]`, build a dict grouping words by their first letter using a *loop* — then convince yourself it can't be done cleanly as a single dict comprehension (why?).
3. Replace this with a generator expression and explain the memory difference:
   ```python
   total = sum([line.count(",") for line in open("big_file.csv")])
   ```

In [10]:
# your solutions here


**Takeaways**

- Comprehensions: `[expr for x in xs if cond]` — list/dict/set variants.
- Conditional expression in the value (`a if c else b`) is different from a filter (`if c` at the end).
- Use generator expressions `(...)` when you only need to consume — `sum`, `any`, `all`, `max`, file iteration.
- `enumerate`, `zip(..., strict=True)`, and unpacking with `*` make most loops one line shorter.

**You're done with `01_data_structures`.** Tackle the exercises in [README](README.md), then say **next learning** for `02_functions_deep`.